In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# shared style
TEMPLATE   = "plotly_white"
COLOR_SEQ  = px.colors.qualitative.Set2
ACCENT     = "#1D9E75"      # teal primary
ACCENT2    = "#534AB7"      # purple secondary
DANGER     = "#D85A30"      # coral for cost
NEUTRAL    = "#888780"      # gray for reference lines

print("Imports OK")

Imports OK


In [ ]:
sheet_id3 = "1ezqBYWhEoTDBbmlIkcZUKYzkc6xixExoRb_XuUU_Ndg"
name = "earnings_clean"
DATA_PATH = f"https://docs.google.com/spreadsheets/d/{sheet_id3}/gviz/tq?tqx=out:csv&sheet={name}"


In [ ]:
df = pd.read_csv(DATA_PATH)

# infer key columns
# expected schema
HOURLY_COL      = "hourly_earnings"       # numeric, dollars per hour
MARKET_COL      = "market"                # categorical string
VEHICLE_COL     = "vehicle_class"         # categorical string
TIME_COL        = "time_of_day"           # categorical: morning/lunch/dinner/late
URBAN_COL       = "urban_density_index"   # numeric
GAS_COL         = "gas_cost_per_mile"     # numeric
PROFIT_COL      = "net_profit_per_hour"   # numeric (after fuel)
MILES_COL       = "miles_per_delivery"    # numeric
TIPS_COL        = "tip_rate"              # numeric (fraction)
FIPS_COL        = "county_fips"           # string / int

print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
print(df.dtypes)
df.head(3)

Rows: 2,000  |  Columns: 35
session_id                     int64
region                        object
census_region                 object
time_slot                     object
driver_experience_tier        object
vehicle_type                  object
vehicle_class                 object
vehicle_est_mpg              float64
hours_worked                 float64
active_hours                 float64
inactive_hours               float64
active_ratio_pct             float64
total_deliveries               int64
deliveries_per_hour          float64
avg_delivery_duration_min    float64
acceptance_rate_pct          float64
total_miles_driven           float64
avg_miles_per_delivery       float64
doordash_pay_usd             float64
gross_base_pay_usd           float64
gross_peak_pay_usd           float64
challenge_bonus_usd          float64
base_pay_per_delivery        float64
has_peak_pay                    bool
has_challenge_bonus             bool
gross_tips_usd               float64
tip_per_de

,session_id,region,census_region,time_slot,driver_experience_tier,vehicle_type,vehicle_class,vehicle_est_mpg,hours_worked,active_hours,...,gross_tips_usd,tip_per_delivery,tip_rate_pct,resolved_mpg,uses_fuel,avg_gas_price,gas_cost_usd,gross_earnings_usd,net_earnings_usd,net_hourly_rate
0,1,"Houston, TX",South,Weekday Midday (2pm-5pm),Experienced (1-2 yr),SUV,SUV,24.0,3.43,2.86,...,19.75,3.95,100,23.0,True,3.25,4.561304,37.90,33.338696,9.719736
1,2,"Miami, FL",South,Weekday Lunch (11am-2pm),Developing (1-3 mo),Sedan,Compact/Midsize,30.0,2.91,1.98,...,11.45,2.29,100,23.0,True,3.25,3.447826,58.77,55.322174,19.011056
2,3,"Atlanta, GA",South,Weekend Lunch (11am-2pm),New (< 1 month),SUV,SUV,24.0,5.82,3.89,...,17.15,3.43,100,23.0,True,3.25,1.533152,24.92,23.386848,4.018359


In [ ]:
rng = np.random.default_rng(42)
plot_df = df[["net_hourly_rate", "region", "vehicle_class"]].copy().dropna(subset=["net_hourly_rate"])
plot_df["jitter"]    = plot_df["net_hourly_rate"] + rng.normal(0, 0.08, len(plot_df))
plot_df["row_index"] = range(len(plot_df))

fig = px.scatter(
    plot_df,
    x="jitter",
    y="row_index",
    color="region",
    color_discrete_sequence=COLOR_SEQ,
    opacity=0.40,
    template=TEMPLATE,
    title="All 2,000 observations — net hourly rate distribution",
    labels={"jitter": "Net hourly rate ($/hr, ±jitter)", "row_index": "Count (observation index)"},
)
fig.update_traces(marker_size=4)
median_val = plot_df["net_hourly_rate"].median()
fig.add_vline(x=median_val, line_dash="dash", line_color=NEUTRAL,
              annotation_text=f"Median ${median_val:.2f}", annotation_position="top right")
fig.update_layout(height=520, legend_title="Region")
fig.show()

In [ ]:
earnings = df["net_hourly_rate"].dropna()
kde_x = np.linspace(earnings.min(), earnings.max(), 300)
kde_y = stats.gaussian_kde(earnings)(kde_x)
kde_y_scaled = kde_y * len(earnings) * (earnings.max() - earnings.min()) / 40

fig = go.Figure()
fig.add_trace(go.Histogram(x=earnings, nbinsx=40, name="Count",
                            marker_color=ACCENT, opacity=0.70))
fig.add_trace(go.Scatter(x=kde_x, y=kde_y_scaled, mode="lines",
                          line=dict(color=ACCENT2, width=2.5), name="KDE"))

for pct, label in [(10,"P10"),(25,"P25"),(75,"P75"),(90,"P90")]:
    v = np.percentile(earnings, pct)
    fig.add_vline(x=v, line_dash="dot", line_color=NEUTRAL, line_width=1,
                  annotation_text=label, annotation_position="top")

fig.update_layout(title="Net hourly rate distribution with KDE",
                  xaxis_title="Net hourly rate ($/hr)", yaxis_title="Count",
                  template=TEMPLATE, height=420, bargap=0.05,
                  legend=dict(orientation="h", y=1.05))
fig.show()
print(earnings.describe().round(2))

count    2000.00
mean       11.59
std         6.86
min         0.52
25%         6.86
50%        10.13
75%        14.70
max        62.75
Name: net_hourly_rate, dtype: float64


In [ ]:
order = (df.groupby("region")["net_hourly_rate"].median()
           .sort_values(ascending=False).index.tolist())

fig = px.box(
    df.dropna(subset=["net_hourly_rate","region"]),
    x="region", y="net_hourly_rate",
    category_orders={"region": order},
    color="region", color_discrete_sequence=COLOR_SEQ,
    template=TEMPLATE,
    title="Net hourly rate by region (sorted by median)",
    labels={"net_hourly_rate":"Net hourly rate ($/hr)", "region":"Region"},
    points="outliers",
)
fig.update_layout(height=440, showlegend=False)
fig.show()

In [ ]:
fig = px.violin(
    df.dropna(subset=["net_hourly_rate","vehicle_class"]),
    x="vehicle_class", y="net_hourly_rate",
    color="vehicle_class", color_discrete_sequence=COLOR_SEQ,
    box=True, points="outliers",
    template=TEMPLATE,
    title="Earnings distribution by vehicle class",
    labels={"net_hourly_rate":"Net hourly rate ($/hr)", "vehicle_class":"Vehicle class"},
)
fig.update_layout(height=440, showlegend=False)
fig.show()

In [ ]:
pivot = (df.groupby(["region","time_slot"])["net_hourly_rate"]
           .median().unstack(fill_value=np.nan))

time_order = ["morning","lunch","dinner","late_night"]
time_order = [t for t in time_order if t in pivot.columns]
if time_order:
    pivot = pivot[time_order]

fig = px.imshow(
    pivot,
    color_continuous_scale="Teal",
    aspect="auto",
    template=TEMPLATE,
    title="Median net hourly rate ($/hr) — region × time slot",
    labels=dict(color="$/hr"),
    text_auto=".1f",
)
fig.update_layout(height=420)
fig.show()

In [ ]:
sub = df[["gross_earnings_usd","net_earnings_usd","uses_fuel","vehicle_class"]].dropna()
sub["fuel_label"] = sub["uses_fuel"].map({True:"Uses fuel", False:"No fuel cost"})

fig = px.scatter(
    sub, x="gross_earnings_usd", y="net_earnings_usd",
    color="fuel_label",
    color_discrete_map={"Uses fuel": DANGER, "No fuel cost": ACCENT},
    opacity=0.45, template=TEMPLATE,
    title="Gross vs net earnings — fuel cost drag by vehicle type",
    labels={"gross_earnings_usd":"Gross earnings ($)", "net_earnings_usd":"Net earnings ($)"},
)
# Perfect 45° line (no cost)
max_val = sub["gross_earnings_usd"].max()
fig.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val], mode="lines",
                          line=dict(color=NEUTRAL, dash="dash", width=1.5),
                          name="No cost (reference)"))
fig.update_traces(marker_size=4, selector=dict(mode="markers"))
fig.update_layout(height=460, legend_title="")
fig.show()

In [ ]:
sub = df[df["uses_fuel"]==True][["avg_gas_price","net_hourly_rate","vehicle_class","gas_cost_usd"]].dropna()

fig = px.scatter(
    sub, x="avg_gas_price", y="net_hourly_rate",
    color="vehicle_class", size="gas_cost_usd",
    color_discrete_sequence=COLOR_SEQ,
    trendline="ols", trendline_scope="overall",
    trendline_color_override=DANGER,
    opacity=0.50, template=TEMPLATE,
    title="Net hourly rate vs average gas price — fuel-using vehicles only",
    labels={"avg_gas_price":"Avg gas price ($/gal)", "net_hourly_rate":"Net hourly rate ($/hr)",
            "gas_cost_usd":"Total gas cost ($)"},
)
fig.add_hline(y=0, line_dash="dash", line_color=DANGER,
              annotation_text="Break-even", annotation_position="right")
fig.update_layout(height=460)
fig.show()

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=["session_id","tip_rate_pct"], errors="ignore")
corr = numeric_df.corr().round(2)

fig = px.imshow(
    corr,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    aspect="auto",
    template=TEMPLATE,
    title="Pearson correlation matrix — all numeric features",
    text_auto=True,
)
fig.update_layout(height=700)
fig.show()

print("\nTop correlations with net_hourly_rate:")
print(corr["net_hourly_rate"].drop("net_hourly_rate").abs()
        .sort_values(ascending=False).head(10).to_string())


Top correlations with net_hourly_rate:
net_earnings_usd             0.65
doordash_pay_usd             0.63
gross_earnings_usd           0.62
challenge_bonus_usd          0.61
deliveries_per_hour          0.48
tip_per_delivery             0.45
avg_delivery_duration_min    0.41
gross_tips_usd               0.40
base_pay_per_delivery        0.31
gross_peak_pay_usd           0.27


In [ ]:
q_df = (df.groupby("region")["net_hourly_rate"]
          .quantile([0.10, 0.50, 0.90])
          .unstack()
          .rename(columns={0.10:"p10", 0.50:"median", 0.90:"p90"})
          .sort_values("median", ascending=True)
          .reset_index())

fig = go.Figure()
fig.add_trace(go.Bar(
    y=q_df["region"], x=q_df["p90"]-q_df["p10"],
    base=q_df["p10"], orientation="h",
    name="P10–P90 range", marker_color=ACCENT, opacity=0.35,
))
fig.add_trace(go.Scatter(
    y=q_df["region"], x=q_df["median"],
    mode="markers", name="Median",
    marker=dict(color=ACCENT2, size=11, symbol="diamond"),
))
fig.update_layout(
    title="Earnings spread by region — P10 / median / P90",
    xaxis_title="Net hourly rate ($/hr)",
    yaxis_title="Region",
    template=TEMPLATE, height=460,
    legend=dict(orientation="h", y=1.05)
)
fig.show()

In [ ]:
sub = df.dropna(subset=["tip_per_delivery","region","driver_experience_tier"])
order = (sub.groupby("driver_experience_tier")["tip_per_delivery"]
            .median().sort_values(ascending=False).index.tolist())

fig = px.box(
    sub,
    x="driver_experience_tier", y="tip_per_delivery",
    color="region",
    category_orders={"driver_experience_tier": order},
    color_discrete_sequence=COLOR_SEQ,
    template=TEMPLATE,
    title="Tip per delivery by experience tier and region",
    labels={"tip_per_delivery":"Tip per delivery ($)", "driver_experience_tier":"Experience tier"},
    points=False,
)
fig.update_layout(height=460, legend_title="Region")
fig.show()

In [ ]:
sub = df[["active_ratio_pct","net_hourly_rate","driver_experience_tier","total_deliveries"]].dropna()

fig = px.scatter(
    sub,
    x="active_ratio_pct", y="net_hourly_rate",
    color="driver_experience_tier",
    size="total_deliveries",
    color_discrete_sequence=COLOR_SEQ,
    trendline="ols", trendline_scope="overall",
    trendline_color_override=DANGER,
    opacity=0.45, template=TEMPLATE,
    title="Active ratio % vs net hourly rate — sized by total deliveries",
    labels={"active_ratio_pct":"Active ratio (%)", "net_hourly_rate":"Net hourly rate ($/hr)",
            "total_deliveries":"Total deliveries", "driver_experience_tier":"Experience tier"},
)
fig.update_traces(marker_size=5, selector=dict(mode="markers"))
fig.update_layout(height=480)
fig.show()